# 06 · VIBE Coding Showcase — The Pricing Logic Explainer Agent

**Workshop:** AI for Actuaries — From Foundations to AI Agents
**Session / Part:** S2.P4
**Slides covered:** S2.P4.5 – S2.P4.16
**Author:** Dr Rohan Yashraj Gupta (FIA, FIAI), with Satya Sai Mudigonda and Sai Krishna Vadali

## What this notebook does

We build, break, and fix an **AI agent** that sits next to Priya Nair, ABC General's pricing actuary, and answers her colleagues' questions about why a motor policy was priced the way it was. Three custom tools, one Agno agent, one Gemini model. We watch it hallucinate a non-existent factor, add a guardrail tool, and watch it behave. Then we adapt the same pattern in five lines for ABC Health and ABC Life.

## Prerequisites

- Google account (for Colab).
- A Gemini API key — free-tier is sufficient. Add it to Colab Secrets as `GEMINI_API_KEY` (key icon in the left sidebar).
- No local install required.

## How to run

Top menu → Runtime → Run all. The first cell installs dependencies; subsequent cells run without intervention.


## 0 · Install dependencies


In [1]:
# Pinned versions, tested on a clean Colab CPU runtime.
# Confirm the exact `agno` version against the workshop GitHub repo at runtime.
%pip install -q agno google-genai


In [2]:
# === Standard imports ===
import os
import json
from typing import Literal
from IPython.display import Markdown, display

import warnings
import pandas as pd

# Reproducibility
SEED = 42

# Display
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


# Suppress notebook / streaming warnings during the demo
warnings.filterwarnings("ignore")


## 1 · Set up the Gemini API


In [3]:
# Gemini API key handling.
# In Colab, store your key in Colab Secrets (left sidebar key icon)
# under the name GOOGLE_API_KEY. The notebook will read it from there.

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    if "GOOGLE_API_KEY" not in os.environ:
        raise RuntimeError(
            "GOOGLE_API_KEY is not set. "
            "Add it via Colab Secrets or set the env variable."
        )

# Pin the exact Gemini model. We use Flash for the workshop (fast, cheap, free-tier-friendly).
MODEL_ID = "gemini-3.1-flash-lite"
print(f"API key loaded. Using model: {MODEL_ID}")


API key loaded. Using model: gemini-3.1-flash-lite


## 2 · The three tools

Our agent isn't allowed to invent rating logic. It can only use what its tools tell it. Three tools, in order:

1. `load_rating_table()` — gives the agent the official ABC Motor rating table.
2. `explain_factor(factor_name, factor_value)` — turns one factor + value into plain English using Gemini.
3. `generate_doc(rating_table, explanations)` — assembles a Markdown commentary document.

Each tool is a normal Python function. Agno introspects the signature and docstring to build the tool schema for Gemini.


### Tool 1 — `load_rating_table`


In [4]:
def load_rating_table() -> dict:
    """
    Loads the ABC Motor private car pricing table.

    This tool returns a simplified motor insurance tariff used for
    illustrative pricing examples in the workshop.

    The table contains:
    - A base premium
    - Several pricing factors
    - Relativity values for each factor level

    The relativity values are multiplicative adjustments applied
    to the base premium.

    Example:
    - A relativity above 1.00 increases premium
    - A relativity below 1.00 decreases premium

    The available pricing factors are:
    - vehicle_age
    - vehicle_type
    - region

    Returns:
        dict:
            Structured pricing table containing:
            - base_premium
            - factors
            - relativity values for each factor level
    """

    return {
        "base_premium": 6500,

        "factors": {
            "vehicle_age": {
                "0-2 years": 0.85,
                "3-5 years": 1.00,
                "6+ years": 1.20,
            },

            "vehicle_type": {
                "Hatchback": 0.90,
                "Sedan": 1.00,
                "SUV": 1.25,
            },

            "region": {
                "Metro": 1.15,
                "Non-Metro": 0.95,
            },
        },
    }

### Tool 2 — `explain_factor`


In [5]:
from google import genai

client = genai.Client()

MODEL_ID = "gemini-3.1-flash-lite"

def explain_factor(factor_name: str, factor_value: str) -> str:
    """
    Explains why a pricing factor changes the insurance premium.

    This tool converts technical pricing logic into simple
    business-friendly language suitable for:
    - underwriting discussions,
    - pricing documentation,
    - management presentations,
    - and non-technical stakeholders.

    The tool:
    1. Reads the pricing relativity from the rating table
    2. Sends the factor information to Gemini
    3. Generates a short natural-language explanation

    Example:
    - Older vehicles may have higher repair frequency
    - SUVs may have larger average claim costs
    - Metro regions may experience heavier traffic exposure

    Args:
        factor_name (str):
            Name of the pricing factor.
            Example: "vehicle_age"

        factor_value (str):
            Selected factor level.
            Example: "6+ years"

    Returns:
        str:
            A short explanation in plain English describing
            why the factor impacts insurance premium.
    """

    rating_table = load_rating_table()

    relativity = rating_table["factors"][factor_name][factor_value]

    prompt = f"""
    Explain this motor insurance pricing factor in simple business language.

    Factor: {factor_name}
    Value: {factor_value}
    Relativity: {relativity}

    Keep the explanation under 60 words.
    """
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
    )

    return response.text.strip()

### Tool 3 — `generate_doc`


In [13]:
def generate_doc(rating_table: dict, explanations: dict) -> str:
    """
    Generates a complete markdown pricing commentary document for an
    ABC Motor insurance policy.

    This tool combines:
    1. The structured pricing table returned by `load_rating_table`
    2. The natural-language explanations returned by `explain_factor`

    The purpose of this tool is to create a final business-friendly
    pricing commentary document suitable for:
    - pricing committee discussions,
    - underwriting reviews,
    - internal documentation,
    - audit trails,
    - and workshop demonstrations.

    The generated markdown document should contain:
    - A document title
    - The base premium
    - A summary of all rating factors and relativities
    - Plain-English explanations for the selected pricing factors

    Expected structure of `rating_table`:
    {
        "base_premium": 6500,
        "factors": {
            "vehicle_age": {
                "0-2 years": 0.85,
                "3-5 years": 1.00
            },
            "vehicle_type": {
                "SUV": 1.25
            }
        }
    }

    Expected structure of `explanations`:
    {
        "vehicle_age:6+ years": "Older vehicles may experience...",
        "vehicle_type:SUV": "SUVs typically have..."
    }

    Important rules:
    - Ignore malformed or unexpected entries safely
    - Convert all output content to strings before assembling markdown
    - Never raise exceptions for missing or invalid fields
    - Return a valid markdown document even if some sections are incomplete

    Args:
        rating_table (dict):
            Structured pricing table generated by `load_rating_table`.

        explanations (dict):
            Dictionary mapping factor identifiers to plain-English
            explanations generated by `explain_factor`.

    Returns:
        str:
            A complete markdown pricing commentary document.
    """

    lines = []

    lines.append("# ABC Motor Pricing Commentary")
    lines.append("")

    base_premium = rating_table.get("base_premium", "Unknown")

    lines.append(f"Base Premium: ₹{base_premium}")
    lines.append("")

    lines.append("## Rating Factors")
    lines.append("")

    factors = rating_table.get("factors", {})

    for factor_name, factor_values in factors.items():

        if not isinstance(factor_values, dict):
            continue

        lines.append(f"### {factor_name}")

        for value, relativity in factor_values.items():
            lines.append(f"- {value}: {relativity}")

        lines.append("")

    lines.append("## Explanations")
    lines.append("")

    for key, explanation in explanations.items():

        lines.append(f"### {str(key)}")
        lines.append(str(explanation))
        lines.append("")

    return "\n".join(lines)

## 3 · Build the agent

Now we glue the tools together with an Agno agent and a Gemini brain. The system prompt is the contract — it tells the agent who it is, who it serves, and what it must not do.


In [7]:
SYSTEM_PROMPT = """
You are a motor insurance pricing assistant for ABC General Insurance.

Your role is to explain how motor insurance premiums are determined using
the official ABC Motor rating table.

Workflow:
1. Always call `load_rating_table` first.
2. Identify the relevant pricing factors for the policy.
3. Use `explain_factor` to generate simple explanations.
4. Use `generate_doc` to create the final markdown report.

Rules:
- Only use factors present in the rating table.
- Never invent pricing factors or relativities.
- Keep explanations clear and business-friendly.
- Return the final response as a markdown document.
"""

print(SYSTEM_PROMPT)


You are a motor insurance pricing assistant for ABC General Insurance.

Your role is to explain how motor insurance premiums are determined using
the official ABC Motor rating table.

Workflow:
1. Always call `load_rating_table` first.
2. Identify the relevant pricing factors for the policy.
3. Use `explain_factor` to generate simple explanations.
4. Use `generate_doc` to create the final markdown report.

Rules:
- Only use factors present in the rating table.
- Never invent pricing factors or relativities.
- Keep explanations clear and business-friendly.
- Return the final response as a markdown document.



In [14]:
from agno.agent import Agent
from agno.models.google import Gemini

# Create the pricing agent
pricing_agent = Agent(
    name="Pricing Logic Explainer",
    model=Gemini(
        id=MODEL_ID
    ),
    tools=[
        load_rating_table,
        explain_factor,
        generate_doc,
    ],
    instructions=[
        SYSTEM_PROMPT
    ],
    markdown=True,
)

print("✅ Pricing agent ready")

✅ Pricing agent ready


## 4 · First run — a well-formed question


In [15]:
query = """
A colleague wants to understand how an ABC Motor premium was determined.

Policy details:
- Vehicle age: 7 years
- Vehicle type: SUV
- Region: Metro

Tasks:
1. Load the official rating table
2. Identify the applicable rating factors
3. Explain each factor in simple business language
4. Generate a markdown pricing commentary document
"""

pricing_agent.print_response(query, stream=True, show_full_reasoning=False)

# ============================================================
# FINAL CLEAN MARKDOWN OUTPUT
# ============================================================

# response = pricing_agent.run(query)

# print("\n")
# print("=" * 70)
# print("📋 GEMINI MODEL RESPONSE")
# print("=" * 70)

# display(Markdown(response.content))

# print("\n" + "=" * 70)
# print("END OF MODEL RESPONSE")
# print("=" * 70)

Output()

## 5 · Now break it — an unknown factor

Watch what happens when a colleague asks about a factor that does not exist in our rating table. Without a guardrail, the LLM cheerfully invents one. This is the failure mode actuaries cannot ship.


In [16]:
hostile_query = (
    "Explain the rating logic for an ABC Motor policy on a 7-year-old SUV "
    "in Tier 2 with NCB 35%, and also tell me how the air-filter discount applies."
)

# ============================================================
# FINAL CLEAN MARKDOWN OUTPUT
# ============================================================

response = pricing_agent.run(query)

print("\n")
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(response.content))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)




📋 GEMINI MODEL RESPONSE


# ABC Motor Pricing Commentary

**Base Premium:** ₹6,500

This document outlines the pricing structure applied to your policy based on the ABC Motor rating table. The final premium is calculated by applying the relevant relativities to the base premium.

## Applied Rating Factors

| Factor Category | Selected Level | Relativity |
| :--- | :--- | :--- |
| **Vehicle Age** | 6+ years | 1.20 |
| **Vehicle Type** | SUV | 1.25 |
| **Region** | Metro | 1.15 |

## Detailed Explanations

### Vehicle Age (6+ years)
This factor indicates that vehicles aged six years or older are considered higher risk. The 1.2 relativity means these vehicles are charged a 20% premium increase compared to the base rate. This adjustment accounts for higher repair costs due to parts scarcity, increased likelihood of mechanical failure, and lower safety technology standards in older models.

### Vehicle Type (SUV)
The vehicle type factor accounts for the specific risk profile of an SUV. A relativity of 1.25 means that SUVs are priced 25% higher than the base model. This adjustment reflects the higher costs associated with SUV repairs, parts, and the increased damage they may cause to other vehicles or property in a collision.

### Region (Metro)
The Metro region factor, with a relativity of 1.15, means that vehicles based in metropolitan areas are charged a 15% premium increase compared to the base rate. This accounts for the higher risk of accidents, theft, and traffic congestion typically found in densely populated city environments.


END OF MODEL RESPONSE


## 6 · The fix — add a guardrail tool

The agent needs an explicit way to check whether a factor exists in the table **before** it tries to explain it. We add a guardrail, register it. Same agent shape, one extra check.


In [38]:
def check_factor_in_table(factor_name: str) -> dict:
    """
    Validates whether a pricing factor exists in the official
    ABC Motor rating table.

    This tool acts as a guardrail for the pricing agent.

    Before explaining any pricing factor, the agent should call
    this tool to confirm that the factor is part of the approved
    tariff structure.

    The purpose of this tool is to prevent:
    - hallucinated pricing factors,
    - invented relativities,
    - unsupported underwriting logic,
    - and inaccurate pricing explanations.

    Example valid factors:
    - vehicle_age
    - vehicle_type
    - region

    If a factor does not exist:
    - the agent should clearly tell the user,
    - should not invent pricing logic,
    - and should stop further explanation for that factor.

    Args:
        factor_name (str):
            Name of the pricing factor to validate.

    Returns:
        dict:
            Dictionary containing:
            - exists (bool):
                Whether the factor exists in the rating table.

            - valid_factors (list[str]):
                List of all approved pricing factors available
                in the ABC Motor tariff.
    """

    rating_table = load_rating_table()

    valid_factors = list(
        rating_table["factors"].keys()
    )

    return {
        "exists": factor_name in valid_factors,
        "valid_factors": valid_factors,
    }

In [39]:
# Updated system prompt with guardrail instructions
SYSTEM_PROMPT_V2 = """
You are a motor insurance pricing assistant for ABC General Insurance.

Your role is to explain how motor insurance premiums are determined
using the official ABC Motor rating table.

Workflow:
1. Load the rating table
2. Check whether each factor exists in the tariff
3. Explain only valid factors
4. Generate a markdown pricing commentary document

Rules:
- Always call `check_factor_in_table` before `explain_factor`
- Never invent pricing factors or relativities
- If a factor is invalid, clearly say so
- List the valid factors available in the tariff
- Keep explanations concise and business-friendly
"""

# Create the guardrailed agent
pricing_agent_v2 = Agent(
    name="Pricing Logic Explainer",

    model=Gemini(
        id=MODEL_ID
    ),

    tools=[
        load_rating_table,
        check_factor_in_table,
        explain_factor,
        generate_doc,
    ],

    instructions=[
        SYSTEM_PROMPT_V2
    ],
    markdown=True,
)

print("✅ Guardrailed pricing agent ready")

✅ Guardrailed pricing agent ready


In [40]:
# Same hostile query, new agent.
# pricing_agent_v2.print_response(hostile_query, stream=True)

# ============================================================
# FINAL CLEAN MARKDOWN OUTPUT
# ============================================================

response = pricing_agent_v2.run(hostile_query)

print("\n")
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(response.content))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)




📋 GEMINI MODEL RESPONSE


- Do not add any text before or after the JSON

For your request regarding the 7-year-old SUV, here is the status of the requested factors:

*   **Vehicle Age (7 years old):** Valid (Factor: `6+ years`)
*   **Vehicle Type (SUV):** Valid
*   **Region (Tier 2):** The current tariff table uses "Region" (categorized as Metro/Non-Metro). "Tier 2" is not a recognized category in our official rating table.
*   **NCB 35%:** This is not currently in the provided rating table factors.
*   **Air-filter discount:** This is **not a valid pricing factor** in our official ABC Motor rating table. I cannot provide logic or relativities for this.

The valid factors supported in our system are: `vehicle_age`, `vehicle_type`, and `region`.

I have generated the explanations for the valid factors identified:

### Pricing Explanations

*   **Vehicle Age (6+ years):** Vehicles aged six years or older are considered 20% riskier to insure than the base model. This higher relativity reflects the increased likelihood of mechanical failure, difficulty in sourcing older replacement parts, and higher repair costs relative to the vehicle's depreciated market value. Consequently, your premium is adjusted upward by a factor of 1.2.
*   **Vehicle Type (SUV):** For insurance pricing, the "SUV" vehicle type carries a relativity of 1.25. This means SUVs are considered 25% riskier or more expensive to repair than a standard vehicle. Consequently, your base premium is multiplied by 1.25 to reflect the higher potential costs associated with insuring this specific type of vehicle.

Would you like me to generate a formal pricing commentary document based on these factors?


END OF MODEL RESPONSE


In [41]:
from agno.guardrails.base import BaseGuardrail
from agno.exceptions import InputCheckError, CheckTrigger
from agno.run.agent import RunInput


class ValidFactorGuardrail(BaseGuardrail):
    """
    Blocks unsupported pricing factors before the agent runs.
    """

    def check(self, run_input: RunInput) -> None:

        content = run_input.input_content.lower()

        valid_factors = [
            "vehicle age",
            "vehicle type",
            "region",
        ]

        blocked_terms = [
            "air filter",
            "music system",
            "seat colour",
            "sunroof type",
        ]

        invalid_factors = [
            term for term in blocked_terms
            if term in content
        ]

        if invalid_factors:

            raise InputCheckError(
                f"Unsupported pricing factor(s): "
                f"{', '.join(invalid_factors)}. "
                f"Valid factors are: "
                f"{', '.join(valid_factors)}.",
                check_trigger=CheckTrigger.INPUT_NOT_ALLOWED,
            )

    async def async_check(self, run_input: RunInput) -> None:
        """
        Async version required by Agno.
        """

        self.check(run_input)

In [42]:
pricing_agent_v2 = Agent(
    name="Pricing Logic Explainer",
    model=Gemini(
        id=MODEL_ID
    ),
    tools=[
        load_rating_table,
        explain_factor,
        generate_doc,
    ],
    instructions=[
        SYSTEM_PROMPT
    ],
    pre_hooks=[
        ValidFactorGuardrail()
    ],
    markdown=True,
)

In [46]:
# Same hostile query, new agent.
# pricing_agent_v2.print_response(hostile_query, stream=True)

# ============================================================
# FINAL CLEAN MARKDOWN OUTPUT
# ============================================================

response = pricing_agent_v2.run(hostile_query)

print("\n")
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(response.content))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)




📋 GEMINI MODEL RESPONSE


Regarding your inquiry about the ABC Motor insurance policy rating for a 7-year-old SUV, I have analyzed the standard rating table provided by our system.

### Pricing Logic for Your Policy

Based on the official ABC Motor rating table, the following factors are applied to your base premium:

*   **Base Premium:** ₹6,500
*   **Vehicle Age (7 years):** Classified under the "6+ years" category, this factor carries a **1.20** multiplier. Older vehicles are rated higher due to the statistical increase in mechanical issues and the complexity/cost of sourcing replacement parts.
*   **Vehicle Type (SUV):** This carries a **1.25** multiplier. SUVs are subject to higher premiums compared to standard vehicles because their size, weight, and specialized components typically result in higher repair costs in the event of a claim.

*Note: Regarding "Tier 2," "NCB 35%," and "Air-filter discount," these specific attributes are not currently supported in the standard rating table available to me. Please consult with your underwriting department to see if these can be applied as manual overrides or supplementary credits to your final calculated premium.*

***

### Summary of Pricing Factors

| Factor | Level | Relativity | Explanation |
| :--- | :--- | :--- | :--- |
| **Vehicle Age** | 6+ years | 1.20 | Reflects higher statistical risk of mechanical failure and part sourcing costs for older models. |
| **Vehicle Type**| SUV | 1.25 | Accounts for the increased repair costs and potential risks associated with the weight and build of an SUV. |

*This commentary is provided for illustrative purposes based on standard ABC Motor rating guidelines.*


END OF MODEL RESPONSE


## 7 · The same pattern, two more LOBs

The whole agent shape — load reference data, check before explaining, explain in plain English, assemble a document — is LOB-agnostic. Below: the same agent retargeted in five lines for ABC Health, then five lines for ABC Life.


### ABC Health — Claim Severity Explainer (5-line diff)


In [44]:
# Health swap: replace the rating table loader with a severity table loader,
# and the system prompt's persona, role, and product. Everything else is reused.

def load_severity_table_health() -> dict:
    """ABC Health 2024 average severity by procedure category and member age band."""
    return {
        "base_severity_inr": 62000.0,
        "factors": {
            "procedure_category": {"DayCare": 0.55, "Medical": 1.00, "Surgical": 1.65, "ICU": 2.40},
            "member_age_band": {"0-17": 0.70, "18-39": 0.85, "40-59": 1.10, "60+": 1.50},
            "product_tier": {"Bronze": 0.90, "Silver": 1.00, "Gold": 1.15},
        },
    }

# 5-line agent diff:
health_agent = Agent(
    name="Claim Severity Explainer",
    model=Gemini(id=MODEL_ID),
    tools=[load_severity_table_health, check_factor_in_table, explain_factor, generate_doc],
    instructions=SYSTEM_PROMPT_V2.replace("Priya Nair", "Dr Ananya Iyer")
                                  .replace("ABC General Insurance", "ABC Health")
                                  .replace("rating", "severity"),
)
print(f"Health agent ready: {health_agent.name}")


Health agent ready: Claim Severity Explainer


### ABC Life — Mortality Assumption Documenter (5-line diff)


In [45]:
# Life swap: load the mortality assumption set instead.

def load_mortality_assumptions_life() -> dict:
    """ABC Life term mortality assumptions, by issue age band and smoker status."""
    return {
        "base_qx_per_1000": 1.0,  # at age 30, non-smoker
        "factors": {
            "issue_age_band": {"18-29": 0.85, "30-39": 1.00, "40-49": 1.55, "50-65": 2.40},
            "smoker_status": {"NS": 1.00, "S": 2.10, "Unknown": 1.50},
            "uw_route": {"NoMed": 1.20, "TeleMed": 1.05, "FullMed": 1.00},
        },
    }

# 5-line agent diff:
life_agent = Agent(
    name="Mortality Assumption Documenter",
    model=Gemini(id=MODEL_ID),
    tools=[load_mortality_assumptions_life, check_factor_in_table, explain_factor, generate_doc],
    instructions=SYSTEM_PROMPT_V2.replace("Priya Nair", "Vikram Rao")
                                  .replace("ABC General Insurance", "ABC Life")
                                  .replace("rating", "mortality assumption"),
)
print(f"Life agent ready: {life_agent.name}")


Life agent ready: Mortality Assumption Documenter


## Wrap-up

You should now be able to:

- Define a tool as a plain Python function that an Agno agent can call.
- Write a system prompt that constrains the agent to a domain and a behaviour.
- Spot the failure mode where the agent invents content the tools never gave it.
- Add a guardrail tool that closes that failure mode.
- Adapt the same agent shape to a different LOB in roughly five lines of diff.

**Where to next:** open Track 3 of the case study brief for the 2-week build.

**Companion slides:** S2.P4.5 – S2.P4.16 of the deck.
